# Adaptive Thinking

## Table of contents
- [Setup](#setup)
- [Adaptive vs Extended Thinking](#adaptive-vs-extended-thinking)
- [Simple query — does thinking activate?](#simple-query)
- [Complex query — thinking kicks in](#complex-query)
- [Side-by-side comparison](#side-by-side-comparison)
- [Tool use — define tools](#tool-use)
- [Tool use — agentic loop](#agentic-loop)
- [Token usage analysis](#token-usage-analysis)
- [Best practices](#best-practices)

**Adaptive Thinking** (`thinking: { type: 'auto' }`) lets Claude decide *whether* to think at all based on task complexity. Simple queries skip the thinking phase entirely; hard queries engage full reasoning — saving tokens and latency automatically.

## Setup

Ensure `@anthropic-ai/sdk` is installed and `ANTHROPIC_API_KEY` is in your environment.

In [ ]:
// npm install @anthropic-ai/sdk


Import the SDK and create a client. Constants are defined once and reused in every cell.

In [ ]:
import Anthropic from '@anthropic-ai/sdk';
import { load } from 'https://deno.land/std@0.220.0/dotenv/mod.ts';

await load({ envPath: '../.env', export: true });

const MODEL_NAME = 'claude-sonnet-4-6';
const ANTHROPIC_API_KEY = Deno.env.get('ANTHROPIC_API_KEY') ?? '';
const MAX_TOKENS = 8000;
const BUDGET = 5000;

const client = new Anthropic({ apiKey: ANTHROPIC_API_KEY });
console.log('Client ready. Model:', MODEL_NAME);


A small helper that prints every content block and the token usage for any response.

In [ ]:
function printResponse(label: string, response: Anthropic.Message): void {
  console.log(`\n==== ${label} ====`);
  for (const block of response.content) {
    if (block.type === 'thinking') {
      const t = (block as any).thinking as string;
      console.log('[THINKING]', t.length > 300 ? t.slice(0, 300) + '...' : t);
    } else if (block.type === 'redacted_thinking') {
      console.log('[REDACTED THINKING]');
    } else if (block.type === 'text') {
      console.log('[ANSWER]', block.text);
    }
  }
  const u = response.usage;
  console.log(`Tokens — in: ${u.input_tokens}, out: ${u.output_tokens}`);
  console.log('==== END ====\n');
}


## Adaptive vs Extended Thinking

| | Extended (`'enabled'`) | Adaptive (`'auto'`) |
|---|---|---|
| Thinking always triggered | Yes | No — Claude decides |
| Token usage | Full budget may be used every call | Only tokens actually needed |
| Latency on simple queries | Higher | Lower |
| Best for | Uniformly complex tasks | Mixed workloads |
| `temperature` / `top_p` compatible | No | No |
| Minimum `budget_tokens` | 1 024 | 1 024 |

The key difference: with `'enabled'`, asking *"What is 2+2?"* still produces a full thinking block. With `'auto'`, Claude answers directly — no extra overhead.

## Simple query

Send a trivial question. With `type: 'auto'`, we expect Claude to skip the thinking phase and answer immediately.

In [ ]:
const simpleRes = await client.messages.create({
  model: MODEL_NAME,
  max_tokens: MAX_TOKENS,
  thinking: { type: 'auto', budget_tokens: BUDGET },
  messages: [{ role: 'user', content: 'What is the capital of France?' }]
});

const simpleHasThinking = simpleRes.content.some(b => b.type === 'thinking');
console.log('Thinking block present:', simpleHasThinking);
printResponse('Simple Query', simpleRes);


## Complex query

Send a question that requires multi-step reasoning. Claude should now decide to engage thinking before answering.

In [ ]:
const complexRes = await client.messages.create({
  model: MODEL_NAME,
  max_tokens: MAX_TOKENS,
  thinking: { type: 'auto', budget_tokens: BUDGET },
  messages: [{
    role: 'user',
    content: 'Three people check into a hotel room costing $30. They each pay $10. The manager gives $5 back; the bellboy pockets $2 and returns $1 to each guest. Now each paid $9 × 3 = $27, plus $2 = $29. Where is the missing $1?'
  }]
});

const complexHasThinking = complexRes.content.some(b => b.type === 'thinking');
console.log('Thinking block present:', complexHasThinking);
printResponse('Complex Query', complexRes);


## Side-by-side comparison

Send the same complex prompt to both `'enabled'` and `'auto'` in parallel and compare output token counts.

In [ ]:
const prompt = 'A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost? Show your reasoning.';

const [extRes, adpRes] = await Promise.all([
  client.messages.create({
    model: MODEL_NAME, max_tokens: MAX_TOKENS,
    thinking: { type: 'enabled', budget_tokens: BUDGET },
    messages: [{ role: 'user', content: prompt }]
  }),
  client.messages.create({
    model: MODEL_NAME, max_tokens: MAX_TOKENS,
    thinking: { type: 'auto', budget_tokens: BUDGET },
    messages: [{ role: 'user', content: prompt }]
  })
]);

console.log('Extended  — thinking present:', extRes.content.some(b => b.type === 'thinking'), '| output tokens:', extRes.usage.output_tokens);
console.log('Adaptive  — thinking present:', adpRes.content.some(b => b.type === 'thinking'), '| output tokens:', adpRes.usage.output_tokens);


## Tool use — define tools

Define two mock tools (`calculator` and `lookup_country_info`) and a simple dispatcher. These are shared across the next two cells.

In [ ]:
const tools: Anthropic.Tool[] = [
  {
    name: 'calculator',
    description: 'Evaluate a mathematical expression and return the numeric result.',
    input_schema: {
      type: 'object',
      properties: { expression: { type: 'string', description: 'Arithmetic expression, e.g. "3 ** 7 - 500"' } },
      required: ['expression']
    }
  },
  {
    name: 'lookup_country_info',
    description: 'Return capital, population, and continent for a given country.',
    input_schema: {
      type: 'object',
      properties: { country: { type: 'string' } },
      required: ['country']
    }
  }
];

const countryDB: Record<string, object> = {
  Japan:   { capital: 'Tokyo',    population: '125 million', continent: 'Asia' },
  Brazil:  { capital: 'Brasília', population: '215 million', continent: 'South America' },
  Germany: { capital: 'Berlin',   population: '84 million',  continent: 'Europe' }
};

function runTool(name: string, input: Record<string, string>): string {
  if (name === 'calculator') {
    try { return String(Function('"use strict"; return (' + input.expression + ')')()) }
    catch { return 'Error: invalid expression' }
  }
  if (name === 'lookup_country_info') {
    return JSON.stringify(countryDB[input.country] ?? { error: 'Not found' });
  }
  return 'Unknown tool';
}

console.log('Tools defined:', tools.map(t => t.name).join(', '));


## Agentic loop

Run an adaptive-thinking agentic loop. Claude may call tools one or more times before giving a final answer.

> **Important:** when passing the assistant turn back, always include **all** `thinking` and `redacted_thinking` blocks — omitting them causes an API error.

In [ ]:
const conversation: Anthropic.MessageParam[] = [{
  role: 'user',
  content: 'What is the capital of Japan, and what is 3 to the power of 7 minus 500?'
}];

let response = await client.messages.create({
  model: MODEL_NAME, max_tokens: MAX_TOKENS,
  thinking: { type: 'auto', budget_tokens: BUDGET },
  tools,
  messages: conversation
});

console.log('Initial thinking block present:', response.content.some(b => b.type === 'thinking'));


Process tool calls in a loop until `stop_reason` is no longer `'tool_use'`.

In [ ]:
while (response.stop_reason === 'tool_use') {
  // Preserve thinking + tool_use blocks in assistant turn
  const assistantBlocks = response.content.filter(
    b => b.type === 'thinking' || b.type === 'redacted_thinking' || b.type === 'tool_use'
  );
  conversation.push({ role: 'assistant', content: assistantBlocks as any });

  // Execute every requested tool
  const toolResults: Anthropic.ToolResultBlockParam[] = response.content
    .filter(b => b.type === 'tool_use')
    .map(b => {
      const tb = b as Anthropic.ToolUseBlock;
      const result = runTool(tb.name, tb.input as Record<string, string>);
      console.log(`Tool: ${tb.name}(${JSON.stringify(tb.input)}) => ${result}`);
      return { type: 'tool_result', tool_use_id: tb.id, content: result };
    });

  conversation.push({ role: 'user', content: toolResults });

  response = await client.messages.create({
    model: MODEL_NAME, max_tokens: MAX_TOKENS,
    thinking: { type: 'auto', budget_tokens: BUDGET },
    tools,
    messages: conversation
  });
}

printResponse('Adaptive Thinking + Tool Use', response);


## Token usage analysis

Compare output tokens for four prompts of increasing complexity. For simple prompts, adaptive thinking should use noticeably fewer tokens than extended thinking.

In [ ]:
const testPrompts = [
  { label: 'Trivial',  text: 'Say hello in one word.' },
  { label: 'Simple',   text: 'List three primary colours.' },
  { label: 'Medium',   text: 'Explain how HTTPS differs from HTTP in 3 bullet points.' },
  { label: 'Complex',  text: 'A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost? Show full reasoning.' }
];

console.log('Label'.padEnd(10), '| Extended out'.padEnd(16), '| Adaptive out'.padEnd(16), '| Saved');
console.log('-'.repeat(55));


Run each prompt through both modes in parallel and print the results.

In [ ]:
for (const { label, text } of testPrompts) {
  const [ext, adp] = await Promise.all([
    client.messages.create({
      model: MODEL_NAME, max_tokens: MAX_TOKENS,
      thinking: { type: 'enabled', budget_tokens: BUDGET },
      messages: [{ role: 'user', content: text }]
    }),
    client.messages.create({
      model: MODEL_NAME, max_tokens: MAX_TOKENS,
      thinking: { type: 'auto', budget_tokens: BUDGET },
      messages: [{ role: 'user', content: text }]
    })
  ]);

  const saved = ext.usage.output_tokens - adp.usage.output_tokens;
  const pct = ((saved / ext.usage.output_tokens) * 100).toFixed(1);
  console.log(
    label.padEnd(10),
    '|', String(ext.usage.output_tokens).padEnd(14),
    '|', String(adp.usage.output_tokens).padEnd(14),
    '|', `-${saved} (${pct}%)`
  );
}


## Best practices

### When to use `type: 'auto'` (Adaptive)
- Mixed workloads — simple and complex queries in the same pipeline
- Cost-sensitive production — pay for thinking only when it helps
- Latency matters — trivial questions should return fast

### When to use `type: 'enabled'` (Extended)
- All queries require deep reasoning (code review, legal analysis, etc.)
- You need a thinking trace for every single response
- You want predictable, consistent reasoning depth

### Shared constraints
| Constraint | Value |
|---|---|
| Minimum `budget_tokens` | 1 024 |
| Compatible with `temperature` / `top_p` / `top_k` | No — API error |
| Response prefill allowed | No |
| Thinking tokens billed as | Output tokens |
| Must preserve thinking blocks in multi-turn | Yes (including `signature`) |